## Brouillon

In [36]:
%matplotlib inline
import time
from datetime import datetime

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import sys
import tempfile
from pathlib import Path

_racine = Path.cwd()
while not (_racine / "src").exists() and _racine != _racine.parent:
    _racine = _racine.parent
if str(_racine) not in sys.path:
    sys.path.insert(0, str(_racine))


from src import config
from src.bassins import charger_bassins, ForetBassins, localiser, carreaux_amont
from src.meteo import Meteo, cout_appel
from src.hydro import RiverApp

## METEO

In [9]:
HORAIRES = ["precipitation", "temperature_2m", "snow_depth", "shortwave_radiation", "rain",
            "soil_moisture_7_to_28cm", "snowfall", "et0_fao_evapotranspiration", "soil_moisture_28_to_100cm"]
JOURNALIERES = ["temperature_2m_max"]

In [40]:
STATION_CODE = "X031001001"
STATION_NOM = "La Durance à Embrun [La Clapière]"
LAT, LON = 44.555356405, 6.498009964
ALTITUDE_STATION_M = 786.0   # altitude_ref_alti_station (Hub'Eau)

print(f"{STATION_NOM} ({STATION_CODE})\n{LAT}°N, {LON}°E, {ALTITUDE_STATION_M} m")

La Durance à Embrun [La Clapière] (X031001001)
44.555356405°N, 6.498009964°E, 786.0 m


In [8]:
m = Meteo()

In [13]:
bassins = charger_bassins()
foret = ForetBassins.depuis_table(bassins)
zone = int(localiser([LON], [LAT])[0])
carreaux = carreaux_amont(zone, foret)
print(f"{len(carreaux)} carreaux")
carreaux.head()

41 carreaux


,carreau,lon_centre,lat_centre,surface_km2,pourcentage
0,830767,6.558140,44.956062,86.012142,3.776103
1,833347,6.547988,44.885763,85.991154,3.775181
2,835933,6.816074,44.815465,85.966729,3.774109
3,835932,6.676971,44.815465,85.957155,3.773689
4,835931,6.537867,44.815465,85.945852,3.773192


In [11]:
len(carreaux)

61

In [12]:
m.contenu_cache()

,carreau,variable,resolution,debut,fin,mo_carreau
0,825618,et0_fao_evapotranspiration,horaire,2000-01-01,2026-10-03,3.041
1,825618,precipitation,horaire,2000-01-01,2026-10-03,3.041
2,825618,rain,horaire,2000-01-01,2026-10-03,3.041
3,825618,shortwave_radiation,horaire,2000-01-01,2026-10-03,3.041
4,825618,snow_depth,horaire,2000-01-01,2026-10-03,3.041
...,...,...,...,...,...,...
605,848912,snowfall,horaire,2000-01-01,2026-10-03,2.971
606,848912,soil_moisture_28_to_100cm,horaire,2000-01-01,2026-10-03,2.971
607,848912,soil_moisture_7_to_28cm,horaire,2000-01-01,2026-10-03,2.971
608,848912,temperature_2m,horaire,2000-01-01,2026-10-03,2.971


In [37]:
meteo = m.telecharger(carreaux, HORAIRES, JOURNALIERES, "2000-01-01", "2026-10-03")
h, j = meteo["horaire"], meteo["journalier"]

# une valeur précise : carreau 830767, température le 15 janvier 2020 à 12 h UTC
h.loc["2020-01-15 12:00", (830767, "temperature_2m")]

# une série complète (une variable d'un carreau)
h[(830767, "precipitation")]

# toutes les variables d'un carreau
h[830767]

# une variable pour tous les carreaux (une colonne par carreau)
h.xs("precipitation", axis=1, level="variable")

# une période
h.loc["2020-01":"2020-03", (830767, "snow_depth")]

# le journalier, pareil
j.loc["2020-07-01", (830767, "temperature_2m_max")]

np.float32(21.4)

## EAU

In [45]:
ra = RiverApp()                  # fournisseur par défaut : Hub'Eau
stations = ra.stations()         # liste des stations (index = identifiant RiverApp, colonne code = code chez le fournisseur)
ID = stations.query("code == @STATION_CODE").index[0]


In [46]:
ra.contenu_cache()

,id,code,debut,fin,mesures,mo
0,52a0c9cee4b03f58295e6f8a,X031001001,2010-04-02 00:00:00+00:00,2026-10-09 01:56:00+00:00,362355,3.552


In [ ]:
ra.mesures(ID)[]

,flow,level
date,,
2010-04-02 01:00:00+00:00,37.599998,NaN
2010-04-02 02:00:00+00:00,37.200001,NaN
2010-04-02 03:00:00+00:00,36.700001,NaN
2010-04-02 04:00:00+00:00,36.299999,NaN
2010-04-02 05:00:00+00:00,35.900002,NaN
...,...,...
2026-10-10 12:00:00+00:00,30.700001,0.42
2026-10-10 12:15:00+00:00,30.700001,0.42
2026-10-10 12:30:00+00:00,30.500000,0.41


In [ ]:
debut = stations.query("code == 'X031001001'")["plus_ancienne_mesure"].values[0]
fin = np.datetime64('2026-09-30T00:00:00.000000000')